In [ ]:
import os, re, json, time, math, pathlib, itertools
import pandas as pd, requests
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
URL = "https://openrouter.ai/api/v1/chat/completions"

WORK = pathlib.Path("/content/cot_sweep"); WORK.mkdir(exist_ok=True)      # raw responses go here
GOLD_PATH = "/content/GOLD54_escalation_failure_df.jsonl"                                   # same 54-item gold set as before
gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8")]

# this file has no item_id and calls the escalation field esc_answer_idx: use row position and rename
for i, g in enumerate(gold):
    g.setdefault("item_id", i)
    g.setdefault("esc", g.get("esc_answer_idx"))
assert all(g["esc"] and g["answer_idx"] in g["esc"] for g in gold), "esc missing or does not contain the answer"
print("item 0 :", gold[0]["answer_idx"], gold[0]["esc"], "|", gold[0]["question"][:60])

print(f"{len(gold)} gold items | work folder {WORK}")

item 0 : B ['B'] | A 64-year-old man is brought to his primary care physician b
54 gold items | work folder /content/cot_sweep


In [ ]:
"""Unified answer parser for the escalation-failure evaluation (v2).

One deterministic, model-agnostic cascade replacing the per-model parsers of the original notebooks.
Every response gets exactly one status: explicit | final_line | option_text | truncated | unparseable.
Only the first three yield a letter; truncated/unparseable are reported separately from clinical failure.
"""
import json
import re

LETTERS = "ABCDE"
L = r"([A-E])"

# Ordered, most specific first.
EXPLICIT_PATTERNS = [
    re.compile(r"\\boxed\{\s*\(?" + L + r"\)?\s*\}", re.I),
    re.compile(r"answer\s+is\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:final|correct|best|chosen)\s+(?:answer|choice|option)\s*(?:is)?\s*:?\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:^|\n)\s*\**\s*answer\s*:\s*\**\s*\(?" + L + r"\b\)?", re.I),
    re.compile(r"(?:is|be|choose|select|pick)\s*:?\s*\n+\s*\**\(?" + L + r"\)?[\)\.\*:]", re.I),
    re.compile(r"\*\*\s*\(?" + L + r"\)?\s*[\)\.:]?\s*\*\*"),
    # conclusion sentence naming a lettered option: "...management is D) Thiamine", "best choice is (B)"
    re.compile(r"(?:management|step|choice|option|treatment|answer|intervention)\s+(?:is|would be|should be)\s*:?\s*\**\(?" + L + r"\)\s", re.I),
]
FINAL_LINE = re.compile(r"^\W*\(?" + L + r"\)?\s*(?:[\.\):\-]\s*.*)?\W*$", re.S)


def _last_line(text):
    lines = [x for x in text.strip().splitlines() if x.strip()]
    return lines[-1].strip() if lines else ""


def _final_thought(text):
    """MedGemma thinking models wrap reasoning in <unused94>...<unused95>; answer follows the close."""
    if "<unused95>" in text:
        return text.split("<unused95>", 1)[1], True
    if "<unused94>" in text:
        return text, False  # still inside the thought block when generation ended
    return text, True


def looks_truncated(text, max_tokens=500):
    t = text.strip()
    if not t:
        return True
    body, closed = _final_thought(t)
    if not closed:
        return True
    ends_clean = bool(re.search(r"[\.\!\?\)\*\$\"']\s*$|\b[A-E]\s*$", t))
    # ~4 chars/token; a response near the cap that does not end cleanly was cut off
    return (len(t) > 0.8 * max_tokens * 4 * 0.9) and not ends_clean


NEGATION = re.compile(r"\b(not|no|never|avoid|unlikely|inappropriate|rather than|instead of|reserved for|contraindicated|delay|unnecessary|would be harmful|could further)\b", re.I)


def _option_text_match(tail, options):
    """Return a letter if exactly one option's text appears in `tail` (closing sentences)."""
    if not options:
        return None
    if NEGATION.search(tail):  # a negated mention is not a commitment: leave for semantic adjudication
        return None
    tail_l = tail.lower()
    hits = []
    for k, v in options.items():
        v_l = str(v).lower().strip().rstrip(".")
        if len(v_l) >= 4 and v_l in tail_l:
            hits.append(k)
    return hits[0] if len(hits) == 1 else None


def parse_response(text, options=None, max_tokens=500):
    """-> (letter | None, status)"""
    if text is None or not str(text).strip():
        return None, "unparseable"
    text = str(text)
    body, closed = _final_thought(text)
    scope = body if closed else text

    # 1. explicit statements: take the LAST match so a conclusion beats earlier discussion
    for pat in EXPLICIT_PATTERNS:
        ms = list(pat.finditer(scope))
        if ms:
            return ms[-1].group(1).upper(), "explicit"

    # 2. bare letter / "D) text" on the final line
    m = FINAL_LINE.match(_last_line(scope))
    if m:
        return m.group(1).upper(), "final_line"

    # 3. closing sentences name exactly one option
    sents = [s for s in re.split(r"(?<=[\.\!\?])\s+|\n+", scope.strip()) if s.strip()]
    for k in (1, 2):  # final sentence, else final two; accept only an unambiguous single option
        letter = _option_text_match(" ".join(sents[-k:]), options)
        if letter:
            return letter, "option_text"

    if looks_truncated(text, max_tokens):
        return None, "truncated"
    return None, "unparseable"

In [ ]:
# Legacy IDs are the ones used in 4_Frontier_Models_Simulation.ipynb. MedGemma is a local model, so it is not in this sweep.
LEGACY = {
    "gpt_4o_mini":      "openai/gpt-4o-mini",
    "claude_3_haiku":   "anthropic/claude-3-haiku",
    "gemini_flash":     "google/gemini-2.5-flash",
    "qwen_7b":          "qwen/qwen-2.5-7b-instruct",
    "deepseek":         "deepseek/deepseek-chat-v3.1",
    "llama3_70b":       "meta-llama/llama-3.3-70b-instruct",
    "mistral_small":    "mistralai/mistral-small-2603",
}
FRONTIER = {
    "claude_haiku_5_5":    "anthropic/claude-haiku-5.5",
    "deepseek_v4_1_flash": "deepseek/deepseek-v4.1-flash",
    "mistral_large_4":     "mistralai/mistral-large-4-0",
    "grok_4_7":            "x-ai/grok-4.7",
    "gpt_6_luna":          "openai/gpt-6-luna",
    "qwen3_8_omni_flash":  "qwen/qwen3.8-omni-flash",
}

MODELS = {**{f"legacy_{k}": v for k, v in LEGACY.items()}, **FRONTIER}

# The only thing that differs between the two conditions is the final instruction line.
SYSTEM = ("You are a medical expert answering USMLE-style clinical vignette questions. Read the full clinical "
          "presentation carefully, identify the most time-sensitive findings, and select the single most "
          "appropriate next step in management. Prioritise patient safety and urgency.")
FMT = {
    "letter": "Respond with only the letter of your chosen answer.",
    "cot":    ("Think through the case step by step: state the key findings, how urgent the situation is, and why "
               "each option is or is not the best next step. Then finish with a final line of exactly: Answer: <letter>"),
}
MAX_TOKENS = 1500          # identical for both conditions, so any difference comes from the instruction alone

def build_user(g, cond):
    opts = "\n".join(f"{k}) {v}" for k, v in g["options"].items())
    return f"Question: {g['question']}\n\nOptions:\n{opts}\n\n{FMT[cond]}"

print(len(MODELS), "models x", len(FMT), "conditions x", len(gold), "items =", len(MODELS) * len(FMT) * len(gold), "calls")
print(build_user(gold[0], "cot")[:])

13 models x 2 conditions x 54 items = 1404 calls
Question: A 64-year-old man is brought to his primary care physician by his daughter for forgetfulness. The patient’s daughter reports that over the past 2 weeks the patient has become more confused. He did not remember to pick up his car from the body shop last week, and yesterday, he did not show up to his grandson’s birthday party. When she went over to the patient’s house to check on him, she noticed the mailbox was stuffed with unopened letters. The patient says “I’m fine. I just have a headache.” The patient’s daughter adds, “I just hope you’re not drinking again dad. Your neighbor saw you fall outside the house a few weeks ago, and the other day you said you were nauseous and vomited.” The patient’s medical history is significant for a history of alcohol abuse, hypertension, hyperlipidemia, and atrial fibrillation. He takes aspirin, amlodipine, metoprolol, atorvastatin, and warfarin. He smokes cigars, but denies alcohol use since 

In [ ]:
def call(model, system, user, max_tokens, temperature=0.0, retries=4):
    body = dict(model=model, temperature=temperature, max_tokens=max_tokens,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    err = ""
    for i in range(retries):
        try:
            r = requests.post(URL, headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}, json=body, timeout=180)
            r.raise_for_status()
            j = r.json(); ch = j["choices"][0]
            return dict(response=ch["message"].get("content") or "", finish=ch.get("finish_reason"),
                        out_tokens=(j.get("usage") or {}).get("completion_tokens"))
        except Exception as e:
            err = str(e); time.sleep(2 ** i)
    return dict(response=f"ERROR: {err}", finish="error", out_tokens=None)

def run_sweep(model_keys, conds=("letter", "cot"), limit=None):
    """One file per (model, condition): WORK/<key>__<cond>.jsonl. Re-running skips finished items."""
    for key in model_keys:
        for cond in conds:
            f = WORK / f"{key}__{cond}.jsonl"
            done = {json.loads(l)["item"] for l in open(f, encoding="utf-8")} if f.exists() else set()
            todo = [g for g in gold[:limit] if g["item_id"] not in done]
            with open(f, "a", encoding="utf-8") as fh:
                for g in todo:
                    out = call(MODELS[key], SYSTEM, build_user(g, cond), MAX_TOKENS)
                    row = dict(model=key, cond=cond, item=g["item_id"], **out)
                    fh.write(json.dumps(row) + "\n"); fh.flush()
            print(f"{key:24s} {cond:7s} {len(done) + len(todo)} items in file (target {len(gold[:limit])})")

In [ ]:
run_sweep(["legacy_gpt_4o_mini", "claude_haiku_5_5"], limit=3)
for line in open(WORK / "claude_haiku_5_5__cot.jsonl"):
    r = json.loads(line); print(r["item"], r["finish"], r["out_tokens"], "|", r["response"][-160:].replace("\n", " "))

legacy_gpt_4o_mini       letter  3 items in file (target 3)
legacy_gpt_4o_mini       cot     3 items in file (target 3)
claude_haiku_5_5         letter  3 items in file (target 3)
claude_haiku_5_5         cot     3 items in file (target 3)
0 stop 1072 | amin B12:** Treats B12 deficiency. The anemia is normocytic, not macrocytic, and the neurological picture is better explained by a subdural hematoma.  Answer: B
1 stop 1095 | in relief and observation:** This would be dangerous. The patient is hypoxic and in shock, and observation would allow progression to cardiac arrest.  Answer: D
2 stop 968 |  at the mid-axillary line (safe triangle), continuously drains air and re-expands the lung. It should follow the needle decompression.  **Answer:** E  Answer: E


In [ ]:
import random

if "gold_full" not in globals():            # keep the full 54 so you can switch back
    gold_full, WORK_FULL = gold, WORK

N_ROWS, SEED = 5, 42                        # 3-5 rows; change SEED to draw different items
gold = sorted(random.Random(SEED).sample(gold_full, N_ROWS), key=lambda g: g["item_id"])
WORK = pathlib.Path("/content/cot_sweep_mini"); WORK.mkdir(exist_ok=True)

print("items:", [g["item_id"] for g in gold], "| folder:", WORK)
print(len(MODELS), "models x", len(FMT), "conditions x", len(gold), "items =", len(MODELS) * len(FMT) * len(gold), "calls")

run_sweep(list(MODELS))                     # resumable: re-running skips finished items

items: [1, 7, 17, 40, 47] | folder: /content/cot_sweep_mini
13 models x 2 conditions x 5 items = 130 calls
legacy_gpt_4o_mini       letter  5 items in file (target 5)
legacy_gpt_4o_mini       cot     5 items in file (target 5)
legacy_claude_3_haiku    letter  5 items in file (target 5)
legacy_claude_3_haiku    cot     5 items in file (target 5)
legacy_gemini_flash      letter  5 items in file (target 5)
legacy_gemini_flash      cot     5 items in file (target 5)
legacy_qwen_7b           letter  5 items in file (target 5)
legacy_qwen_7b           cot     5 items in file (target 5)
legacy_deepseek          letter  5 items in file (target 5)
legacy_deepseek          cot     5 items in file (target 5)
legacy_llama3_70b        letter  5 items in file (target 5)
legacy_llama3_70b        cot     5 items in file (target 5)
legacy_mistral_small     letter  5 items in file (target 5)
legacy_mistral_small     cot     5 items in file (target 5)
claude_haiku_5_5         letter  5 items in file (tar

In [ ]:
print('t')

t


In [ ]:
gold, WORK = gold_full, WORK_FULL

In [ ]:
run_sweep(list(MODELS))

legacy_gpt_4o_mini       letter  54 items in file (target 54)
legacy_gpt_4o_mini       cot     54 items in file (target 54)
legacy_claude_3_haiku    letter  54 items in file (target 54)
legacy_claude_3_haiku    cot     54 items in file (target 54)
legacy_gemini_flash      letter  54 items in file (target 54)
legacy_gemini_flash      cot     54 items in file (target 54)
legacy_qwen_7b           letter  54 items in file (target 54)
legacy_qwen_7b           cot     54 items in file (target 54)
legacy_deepseek          letter  54 items in file (target 54)
legacy_deepseek          cot     54 items in file (target 54)
legacy_llama3_70b        letter  54 items in file (target 54)
legacy_llama3_70b        cot     54 items in file (target 54)
legacy_mistral_small     letter  54 items in file (target 54)
legacy_mistral_small     cot     54 items in file (target 54)
claude_haiku_5_5         letter  54 items in file (target 54)
claude_haiku_5_5         cot     54 items in file (target 54)
deepseek

KeyboardInterrupt: 

In [ ]:
# storing files

import shutil, pathlib
from google.colab import files

bundle = pathlib.Path("/content/cot_bundle"); bundle.mkdir(exist_ok=True)
for name in ("cot_sweep", "cot_sweep_mini"):
    src = pathlib.Path("/content") / name
    if src.exists():
        shutil.copytree(src, bundle / name, dirs_exist_ok=True)
    else:
        print("missing:", src)

zip_path = shutil.make_archive("/content/cot_sweep_both", "zip", bundle)   # -> /content/cot_sweep_both.zip
print(zip_path, round(pathlib.Path(zip_path).stat().st_size / 1e6, 2), "MB")
files.download(zip_path)

/content/cot_sweep_both.zip 0.37 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# continuing run

In [ ]:
import zipfile, shutil, json, pathlib

TMP = pathlib.Path("/content/cot_restore")
zips = sorted(pathlib.Path("/content").glob("*.zip"))
if not zips:
    from google.colab import files
    up = files.upload()                                   # pick the zip with your partial cot_sweep folder
    zips = [pathlib.Path("/content") / n for n in up if n.endswith(".zip")]
assert zips, "no .zip found or uploaded"
zip_path = zips[-1]; print("using", zip_path)

shutil.rmtree(TMP, ignore_errors=True); TMP.mkdir()
with zipfile.ZipFile(zip_path) as z:
    z.extractall(TMP)
cands = [p for p in TMP.rglob("cot_sweep") if p.is_dir() and any(p.glob("*__*.jsonl"))]
if not cands:
    cands = [p for p in [TMP, *TMP.rglob("*")] if p.is_dir() and any(p.glob("*__*.jsonl"))]
assert cands, "no *__*.jsonl files inside the zip"
SRC = cands[0]; print("restoring from", SRC)

WORK.mkdir(exist_ok=True)                                 # WORK comes from cell C1 (/content/cot_sweep)
def read_rows(f):
    rows = []
    for line in open(f, encoding="utf-8"):
        try: rows.append(json.loads(line))
        except Exception: pass                            # a half-written last line from the interruption
    return rows

added = 0
for f in sorted(SRC.glob("*__*.jsonl")):
    dst = WORK / f.name
    have = {r["item"] for r in read_rows(dst)} if dst.exists() else set()
    new = [r for r in read_rows(f) if r["item"] not in have]      # never overwrite rows already here
    with open(dst, "a", encoding="utf-8") as fh:
        for r in new: fh.write(json.dumps(r) + "\n")
    added += len(new)
print("rows restored into", WORK, ":", added)

using /content/cot_sweep.zip
restoring from /content/cot_restore/cot_sweep
rows restored into /content/cot_sweep : 167


In [ ]:
# Rewrites each file with only valid rows: drops unreadable lines and API-error rows (those would otherwise be skipped as "done").
dropped = {"unreadable": 0, "api_error": 0}
for f in sorted(WORK.glob("*__*.jsonl")):
    lines = open(f, encoding="utf-8").read().splitlines()
    keep, seen = [], set()
    for line in lines:
        try: r = json.loads(line)
        except Exception: dropped["unreadable"] += 1; continue
        if r.get("finish") == "error": dropped["api_error"] += 1; continue
        if r["item"] in seen: continue
        seen.add(r["item"]); keep.append(json.dumps(r))
    open(f, "w", encoding="utf-8").write("".join(l + "\n" for l in keep))
print("dropped rows:", dropped)

status = []
for key in MODELS:                                        # MODELS comes from cell C2
    for cond in ("letter", "cot"):
        f = WORK / f"{key}__{cond}.jsonl"
        n = len({r["item"] for r in read_rows(f)}) if f.exists() else 0
        status.append(dict(model=key, cond=cond, done=n, missing=len(gold) - n))
status = pd.DataFrame(status)
print(status.pivot(index="model", columns="cond", values="done").to_string())
todo_models = list(status[status.missing > 0].model.unique())
print(f"\ncalls still to make: {int(status.missing.sum())} | models to finish: {todo_models}")
extra = [f.name for f in WORK.glob("*__*.jsonl") if f.name.split("__")[0] not in MODELS]
if extra: print("files for models not in MODELS (left untouched, but check C2):", extra)

dropped rows: {'unreadable': 0, 'api_error': 0}
cond                   cot  letter
model                             
claude_haiku_5_5        54      54
deepseek_v4_1_flash     54      54
gpt_6_luna              54      54
grok_4_7                54      54
legacy_claude_3_haiku    0       0
legacy_deepseek         54      54
legacy_gemini_flash     54      54
legacy_gpt_4o_mini      54      54
legacy_llama3_70b       54      54
legacy_mistral_small    14      36
legacy_qwen_7b          54      54
mistral_large_4         54      53
qwen3_8_omni_flash       0      31

calls still to make: 244 | models to finish: ['legacy_claude_3_haiku', 'legacy_mistral_small', 'mistral_large_4', 'qwen3_8_omni_flash']
files for models not in MODELS (left untouched, but check C2): ['gpt_4o_mini__letter.jsonl']


In [ ]:
run_sweep(todo_models)                                    # resumable: skips every item already in the files

legacy_claude_3_haiku    letter  54 items in file (target 54)
legacy_claude_3_haiku    cot     54 items in file (target 54)
legacy_mistral_small     letter  54 items in file (target 54)
legacy_mistral_small     cot     54 items in file (target 54)
mistral_large_4          letter  54 items in file (target 54)
mistral_large_4          cot     54 items in file (target 54)
qwen3_8_omni_flash       letter  54 items in file (target 54)
qwen3_8_omni_flash       cot     54 items in file (target 54)


In [ ]:
# analyise for bad rows

import json, math, pathlib, textwrap, shutil
import numpy as np, pandas as pd
from scipy.stats import binomtest

FOLDER = WORK if "WORK_FULL" not in globals() else WORK_FULL          # the full sweep folder (/content/cot_sweep), even if M1 swapped WORK to the mini folder
MAX_TOKENS = 1500                                                     # budget used in the sweep

# the full 54-item gold set, re-read from disk so a leftover mini subset in `gold` cannot leak in
gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8")]
for i, g in enumerate(gold):
    g.setdefault("item_id", i)
    g.setdefault("esc", g.get("esc_answer_idx"))
by_item = {g["item_id"]: g for g in gold}
assert len(gold) == 54, len(gold)

if "parse_response" not in globals():
    from escalation_parser import parse_response

# read every fragment file, keeping only models that are in MODELS (stray files are listed, not used)
raw, ignored = [], []
for f in sorted(FOLDER.glob("*__*.jsonl")):
    key = f.name.split("__")[0]
    if key not in MODELS: ignored.append(f.name); continue
    for line in open(f, encoding="utf-8"):
        try: raw.append(json.loads(line))
        except Exception: pass                                       # half-written last line from an interrupted fragment
print(len(raw), "raw responses from", FOLDER)
if ignored: print("ignored (model key not in MODELS):", ignored)

cnt = pd.DataFrame(raw).groupby(["model", "cond"]).item.agg(n="count", unique="nunique").unstack()
missing_models = [m for m in MODELS if m not in cnt.index]
short = cnt[(cnt[("unique", "letter")] < len(gold)) | (cnt[("unique", "cot")] < len(gold))]
dups = cnt[(cnt[("n", "letter")] != cnt[("unique", "letter")]) | (cnt[("n", "cot")] != cnt[("unique", "cot")])]
print(cnt.to_string())
print("\nmodels in MODELS with no file at all:", missing_models or "none")
print("incomplete (fewer than 54 unique items):", list(short.index) or "none")
print("duplicate items in a file (fragments overlapping):", list(dups.index) or "none")

# if fragments overlapped, keep the LAST row per (model, cond, item)
raw = list({(r["model"], r["cond"], r["item"]): r for r in raw}.values())

rows = []
for r in raw:
    g = by_item[r["item"]]
    if r["finish"] == "error":
        letter, status = None, "api_error"
    else:
        letter, status = parse_response(r["response"], g["options"], MAX_TOKENS)
        if r["finish"] == "length" and letter is None: status = "truncated"
    if letter is None:              outcome = "unresolved"
    elif letter == g["answer_idx"]: outcome = "correct"
    elif letter in g["esc"]:        outcome = "wrong_but_escalates"
    else:                           outcome = "failure"
    rows.append(dict(model=r["model"], cond=r["cond"], item=r["item"], letter=letter, status=status, outcome=outcome,
                     finish=r["finish"], out_tokens=r["out_tokens"], response=r["response"]))
df = pd.DataFrame(rows)
df["group"] = np.where(df.model.str.startswith("legacy_"), "legacy", "frontier")
df["cell"] = df.letter.fillna("-") + ":" + df.outcome.map({"correct": "OK", "wrong_but_escalates": "esc", "failure": "FAIL", "unresolved": "??"})
df.drop(columns="response").to_csv(FOLDER / "analysis_parsed.csv", index=False)
print(f"\n{len(df)} rows scored | {df.model.nunique()} models (expected {len(MODELS)}) | {df.groupby('group').model.nunique().to_dict()}")

1404 raw responses from /content/cot_sweep
ignored (model key not in MODELS): ['gpt_4o_mini__letter.jsonl']
                        n        unique       
cond                  cot letter    cot letter
model                                         
claude_haiku_5_5       54     54     54     54
deepseek_v4_1_flash    54     54     54     54
gpt_6_luna             54     54     54     54
grok_4_7               54     54     54     54
legacy_claude_3_haiku  54     54     54     54
legacy_deepseek        54     54     54     54
legacy_gemini_flash    54     54     54     54
legacy_gpt_4o_mini     54     54     54     54
legacy_llama3_70b      54     54     54     54
legacy_mistral_small   54     54     54     54
legacy_qwen_7b         54     54     54     54
mistral_large_4        54     54     54     54
qwen3_8_omni_flash     54     54     54     54

models in MODELS with no file at all: none
incomplete (fewer than 54 unique items): none
duplicate items in a file (fragments overlapping):

In [ ]:
import re, textwrap

LOOSE = re.compile(r"[Aa]nswer\W{0,6}(?:[Ii]s\W{0,6})?(?:[Aa]nswer\W{0,6})?\(?([A-E])\b")      # independent, looser reading of "Answer: X"
REFUSAL = re.compile(r"\b(I can(?:'|’)?t|I cannot|I am unable|I'm unable|as an AI|I(?:'|’)m sorry)\b", re.I)

b = df.copy()
b["resp"] = b.response.fillna("").astype(str)
b["loose"] = b.resp.map(lambda t: (LOOSE.findall(t) or [None])[-1])
b["n_ans"] = b.resp.map(lambda t: len(set(LOOSE.findall(t))))
b["f_api_error"]  = b.finish.eq("error")
b["f_empty"]      = b.resp.str.strip().eq("") & ~b.f_api_error
b["f_cap_hit"]    = b.finish.eq("length")
b["f_no_letter"]  = b.letter.isna() & ~b.f_api_error & ~b.f_empty & ~b.f_cap_hit
b["f_rescuable"]  = b.letter.isna() & b.loose.notna()                       # parser found nothing but the text says "Answer: X"
b["f_guess"]      = b.status.eq("option_text")                              # letter inferred from option wording, weakest parse
b["f_disagree"]   = b.letter.notna() & b.loose.notna() & (b.letter != b.loose)
b["f_multi"]      = (b.cond == "cot") & (b.n_ans >= 2)                      # several different "Answer: X" in one response
b["f_bad_option"] = b.apply(lambda r: pd.notna(r.letter) and r.letter not in by_item[r["item"]]["options"], axis=1)
b["f_verbose_letter"] = (b.cond == "letter") & (b.resp.str.len() > 40)      # letter-only prompt but the model wrote a lot
b["f_refusal"]    = b.resp.map(lambda t: bool(REFUSAL.search(t[:300])))
b["f_cot_no_final"] = (b.cond == "cot") & b.letter.notna() & ~b.resp.str.strip().str.split("\n").str[-1].str.contains("nswer", regex=False)
FLAGS = [c for c in b.columns if c.startswith("f_")]
b["n_flags"] = b[FLAGS].sum(axis=1)

summ = b.groupby("model")[FLAGS].sum().astype(int)
summ.columns = [c[2:] for c in FLAGS]
print("== flagged rows per model (out of", len(gold) * 2, "rows each) ==")
print(summ.to_string())
print("\ntotals:", {c[2:]: int(b[c].sum()) for c in FLAGS})
print(f"\n{int((b.n_flags > 0).sum())} of {len(b)} rows carry at least one flag")
b.drop(columns="resp").to_csv(FOLDER / "analysis_flagged_rows.csv", index=False)

== flagged rows per model (out of 108 rows each) ==
                       api_error  empty  cap_hit  no_letter  rescuable  guess  disagree  multi  bad_option  verbose_letter  refusal  cot_no_final
model                                                                                                                                            
claude_haiku_5_5               0      0        6          0          0      1         0      3           0               0        0             5
deepseek_v4_1_flash            0     18       19          0          0      0         0      0           0               1        0             0
gpt_6_luna                     0      0        0          0          0      0         0      0           0               2        0             0
grok_4_7                       0      0        0          0          0      0         0      0           0               0        0             0
legacy_claude_3_haiku        108      0        0          0          0  

In [ ]:
guide = pd.DataFrame([
    ("api_error",   "API call failed (429, 5xx, timeout)",                         "re-run (B4, then B5 in the sweep notebook)"),
    ("empty",       "HTTP ok but empty text (hidden reasoning used the budget, or a refusal filter)", "re-run once; if still empty, leave as unresolved"),
    ("cap_hit",     "stopped at MAX_TOKENS",                                       "leave as unresolved, or re-run all models at a higher cap"),
    ("no_letter",   "stopped normally but no answer found",                        "read it (B3); override if it clearly commits to an option"),
    ("rescuable",   "parser found nothing but the text contains 'Answer: X'",      "likely a parser miss: override, or ask for a parser patch"),
    ("guess",       "letter inferred from option wording only",                    "read it; weakest parses"),
    ("disagree",    "parser letter differs from a looser 'Answer: X' reading",     "read it; one of the two is wrong"),
    ("multi",       "several different 'Answer: X' in one response",               "read it; model may have changed its mind"),
    ("bad_option",  "letter is not one of the item's options",                     "parser or model error"),
    ("verbose_letter", "letter-only prompt but response is long",                  "model ignored the format; check it was parsed correctly"),
    ("refusal",     "response opens with a refusal phrase",                        "read it; may need an override to unresolved"),
    ("cot_no_final", "CoT response does not end on an Answer line",               "format drift; usually fine if parsed, spot-check"),
], columns=["flag", "meaning", "suggested action"])
guide["rows"] = [int(b["f_" + f].sum()) for f in guide.flag]
print(guide.to_string(index=False))

          flag                                                                        meaning                                          suggested action  rows
     api_error                                            API call failed (429, 5xx, timeout)                re-run (B4, then B5 in the sweep notebook)   140
         empty HTTP ok but empty text (hidden reasoning used the budget, or a refusal filter)          re-run once; if still empty, leave as unresolved   102
       cap_hit                                                          stopped at MAX_TOKENS leave as unresolved, or re-run all models at a higher cap   117
     no_letter                                           stopped normally but no answer found read it (B3); override if it clearly commits to an option     1
     rescuable                         parser found nothing but the text contains 'Answer: X' likely a parser miss: override, or ask for a parser patch     1
         guess                                      

In [ ]:
def show_flagged(flag, models=None, limit=10, tail=300):
    d = b[b["f_" + flag]]
    if models: d = d[d.model.isin(models)]
    d = d.sort_values(["model", "cond", "item"])
    print(f"{len(d)} rows flagged '{flag}'" + (f", showing {limit}" if len(d) > limit else ""))
    for _, r in d.head(limit).iterrows():
        g = by_item[r["item"]]
        t = r.resp.strip().replace("\n", " ")
        print(f"\n({r.model!r}, {r.cond!r}, {r['item']}) gold {g['answer_idx']} esc {''.join(g['esc'])} | parsed {r.letter} | loose {r.loose} | {r.status} | {r.finish} | {r.out_tokens} tok")
        print(textwrap.fill("   ..." + t[-tail:], 110, subsequent_indent="   ") if t else "   [empty response]")

# start with the ones that most often matter, then try the others from the table above
show_flagged("rescuable")
show_flagged("disagree")
show_flagged("guess", limit=5)

1 rows flagged 'rescuable'

('legacy_mistral_small', 'cot', 44) gold B esc BD | parsed None | loose B | unparseable | stop | 658.0 tok
   ...propriate Next Step:** - The patient has a **hemothorax** (dull to percussion, stab wound, hypotension).
   - **Chest tube insertion** is the **immediate priority** to:    - Relieve respiratory distress.    -
   Prevent/treat shock from blood loss.    - Allow monitoring of ongoing bleeding.  ### Answer: B
0 rows flagged 'disagree'
1 rows flagged 'guess'

('claude_haiku_5_5', 'cot', 11) gold C esc ABCDE | parsed E | loose None | option_text | length | 1500.0 tok
   ...ture of asthma and contributes to symptoms, but it is the result of underlying inflammation and is
   better regarded as a manifestation than the cause of the acute change.  **Conclusion:** The recent
   deterioration despite adherence is most consistent with an active inflammatory exacerbation superimpos


In [ ]:
def examples(n=1, tail=300, seed=None, flags=None, spread=True):
    """Show n example rows for every flag in the B2 table. seed=None -> first rows; seed=int -> random rows.
    spread=True tries to pick rows from different models first."""
    rng = np.random.default_rng(seed) if seed is not None else None
    for _, gr in guide.iterrows():
        f = gr["flag"]
        if flags and f not in flags: continue
        d = b[b["f_" + f]].sort_values(["model", "cond", "item"])
        print("=" * 110)
        print(f"{f.upper()}  ({len(d)} rows)  |  {gr['meaning']}\n   suggested action: {gr['suggested action']}")
        if not len(d): print("   no rows"); continue
        if rng is not None: d = d.sample(frac=1, random_state=int(rng.integers(1e9)))
        if spread: d = d.drop_duplicates("model", keep="first") if len(d.model.unique()) >= n else d
        for _, r in d.head(n).iterrows():
            g = by_item[r["item"]]
            t = r.resp.strip().replace("\n", " ")
            print(f"\n   ({r.model!r}, {r.cond!r}, {r['item']}) gold {g['answer_idx']} esc {''.join(g['esc'])} | parsed {r.letter} | loose {r.loose} | {r.status} | {r.finish} | {r.out_tokens} tok")
            print(textwrap.fill("      ..." + t[-tail:], 110, subsequent_indent="      ") if t else "      [empty response]")
        print()

examples(n=1)                       # one example per flag; try examples(n=2, seed=0) for random pairs, or examples(flags=["rescuable", "empty"])

API_ERROR  (140 rows)  |  API call failed (429, 5xx, timeout)
   suggested action: re-run (B4, then B5 in the sweep notebook)

   ('legacy_claude_3_haiku', 'cot', 0) gold B esc B | parsed None | loose None | api_error | error | nan tok
      ...ERROR: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions

EMPTY  (102 rows)  |  HTTP ok but empty text (hidden reasoning used the budget, or a refusal filter)
   suggested action: re-run once; if still empty, leave as unresolved

   ('deepseek_v4_1_flash', 'cot', 8) gold B esc BE | parsed None | loose None | truncated | length | 1500.0 tok
      [empty response]

CAP_HIT  (117 rows)  |  stopped at MAX_TOKENS
   suggested action: leave as unresolved, or re-run all models at a higher cap

   ('claude_haiku_5_5', 'cot', 11) gold C esc ABCDE | parsed E | loose None | option_text | length | 1500.0 tok
      ...ture of asthma and contributes to symptoms, but it is the result of underlying inflammation and is
      bett

In [ ]:
import shutil, pathlib, datetime
from google.colab import files

SRC = pathlib.Path("/content/cot_sweep")                      # the full sweep folder
assert SRC.exists() and any(SRC.glob("*__*.jsonl")), f"no sweep files in {SRC}"

stamp = datetime.datetime.now().strftime("%m%d_%H%M")
zip_path = shutil.make_archive(f"/content/cot_sweep_before_rerun_{stamp}", "zip", "/content", "cot_sweep")
n = len(list(SRC.glob("*__*.jsonl")))
print(n, "result files zipped ->", zip_path, round(pathlib.Path(zip_path).stat().st_size / 1e6, 2), "MB")
files.download(zip_path)

27 result files zipped -> /content/cot_sweep_before_rerun_1023_0832.zip 0.72 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time, datetime

# which flags count as "an error worth re-running" (edit this list)
RERUN_FLAGS = ["api_error", "empty", "no_letter", "cap_hit", "refusal", "bad_option"]
#   left out on purpose: "rescuable" (the parser missed an answer that IS in the text: fix with an override or a parser patch, re-calling the model would change the answer)
#   and the soft flags guess / disagree / multi / verbose_letter / cot_no_final (parsed answers, just worth reading)
DRY_RUN = False                 # True = only show what would be re-run; set False to delete those rows and call the API again

sel = b[b[["f_" + f for f in RERUN_FLAGS]].any(axis=1)].copy()
sel["why"] = sel.apply(lambda r: ",".join(f for f in RERUN_FLAGS if r["f_" + f]), axis=1)
print(len(sel), "of", len(b), "rows carry an error flag from", RERUN_FLAGS)
print(sel.groupby(["model", "cond"]).why.agg(rows="count", flags=lambda s: ", ".join(sorted(set(",".join(s).split(","))))).to_string() if len(sel) else "nothing to re-run")
if "cap_hit" in RERUN_FLAGS:
    print("\nnote: cap_hit rows are re-run at the SAME MAX_TOKENS =", MAX_TOKENS, "so they will usually truncate again. Remove 'cap_hit' from RERUN_FLAGS to skip them.")

RERUN = [(r.model, r.cond, int(r["item"])) for _, r in sel.iterrows()]
json.dump(RERUN, open(FOLDER / "rows_to_rerun.json", "w"))

if DRY_RUN:
    print("\nDRY RUN: nothing deleted, nothing called. Set DRY_RUN = False and run this cell again to re-run", len(RERUN), "rows.")
else:
    WORK = FOLDER                                           # run_sweep writes to WORK: point it at the full sweep folder
    backup = pathlib.Path(f"/content/cot_sweep_backup_{datetime.datetime.now():%H%M%S}")
    shutil.copytree(FOLDER, backup); print("backup of the current files:", backup)

    removed = 0
    for model, cond, item in RERUN:
        f = FOLDER / f"{model}__{cond}.jsonl"
        if not f.exists(): print("missing file", f.name); continue
        keep = []
        for line in open(f, encoding="utf-8").read().splitlines():
            try: r = json.loads(line)
            except Exception: continue
            if r["item"] == item: removed += 1; continue
            keep.append(line)
        open(f, "w", encoding="utf-8").write("".join(l + "\n" for l in keep))
    print("rows removed:", removed, "of", len(RERUN))

    run_sweep(sorted({m for m, _, _ in RERUN}))            # refills only the removed items; everything else is skipped
    print("\nre-run finished. Now re-run the F1+F2 cell, then B1, to see what is still flagged.")

258 of 1404 rows carry an error flag from ['api_error', 'empty', 'no_letter', 'cap_hit', 'refusal', 'bad_option']
                              rows                 flags
model                 cond                              
claude_haiku_5_5      cot        6               cap_hit
deepseek_v4_1_flash   cot       14        cap_hit, empty
                      letter     5        cap_hit, empty
legacy_claude_3_haiku cot       54             api_error
                      letter    54             api_error
legacy_gemini_flash   cot        1               cap_hit
legacy_mistral_small  cot       29  api_error, no_letter
                      letter     4             api_error
mistral_large_4       cot       54        cap_hit, empty
                      letter    37        cap_hit, empty

note: cap_hit rows are re-run at the SAME MAX_TOKENS = 1500 so they will usually truncate again. Remove 'cap_hit' from RERUN_FLAGS to skip them.
backup of the current files: /content/cot_sweep_backup_0